In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph,START,END

In [ ]:
class BatsmanState(TypedDict):
    balls: int
    runs: int
    fours:int
    sixes:int

    strike_rate: float
    balls_per_boundary: float
    boundary_percent: float
    summary: str

In [ ]:
def calculate_strike_rate(state:BatsmanState) -> BatsmanState:
    sr = (state['runs']/state['balls'])*100
    # now we will just return the thing that we updated. because in parallel workflows, we send partial state, not full state. to avoid conflicts in the next step. also we donot update the state directly. we just return the updated things.
    return {"strike_rate":sr}

def calculate_balls_per_boundary(state:BatsmanState) -> BatsmanState:
    bpb = state['balls']/(state['fours'] + state['sixes'])
    return {'balls_per_boundary':bpb}

def calculate_boundary_percent(state:BatsmanState) -> BatsmanState:
    bp =(((state['fours'] * 4) + (state['sixes'] * 6))/state['runs']) * 100
    return {'boundary_percent':bp}

def summarize(state:BatsmanState) -> BatsmanState:
    summary = f"""Strike rate is: {state['strike_rate']} \n Balls per boundary: {state['balls_per_boundary']} \n Boundary precent: {state['boundary_percent']}"""
    return {"summary":summary}

In [ ]:
graph = StateGraph(BatsmanState)

# Add nodes
graph.add_node("calculate_strike_rate",calculate_strike_rate)
graph.add_node("calculate_balls_per_boundary",calculate_balls_per_boundary)
graph.add_node("calculate_boundary_percent",calculate_boundary_percent)
graph.add_node("summarize",summarize)

# Add edges
graph.add_edge(START,"calculate_strike_rate")
graph.add_edge(START,"calculate_boundary_percent")
graph.add_edge(START,"calculate_balls_per_boundary")

graph.add_edge("calculate_strike_rate","summarize")
graph.add_edge("calculate_boundary_percent","summarize")
graph.add_edge("calculate_balls_per_boundary","summarize")

graph.add_edge("summarize",END)

workflow = graph.compile()

In [ ]:
initial_state = {
    "runs":100,
    "balls":50,
    "fours":6,
    "sixes": 4
}

final_state = workflow.invoke(initial_state)
print(final_state)